# DeepSeek API Tool Calling 测试

**目的**: 测试 DeepSeek API 的工具调用 (Tool Calling) 功能，并打印详细的请求和响应日志。
协助排查在获取 AI 回复时出现的 "Failed to fetch" 或其他网络/解析错误。

官方文档参考: https://api-docs.deepseek.com/zh-cn/guides/tool_calls

In [ ]:
import os
import json
from openai import OpenAI

# 1. 准备 API Key 和 Base URL
# 注意：这里直接使用了从你环境里读到的 API KEY(如果环境没变可以顺利执行)
API_KEY = "sk-065e5668a86c4730a34ed2b3f5d3f01b" 
BASE_URL = "https://api.deepseek.com"

# 初始化 OpenAI 客户端
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print("✅ 客户端初始化成功")

### 定义 Tool (工具) 相关的 Mock
这里我们需要在发送给 AI 的参数中定义它有权限使用的工具列表，同时也需要在本地有一个同名的函数来实现它。

In [ ]:
# 2. 定义工具 (Tools) 及其对应的本地 Mock 函数
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g. Hangzhou"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_article",
            "description": "Create an article with a specific title and path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article"
                    },
                    "path": {
                        "type": "string",
                        "description": "The path where the article will be saved"
                    },
                    "content": {
                        "type": "string",
                        "description": "The content of the article"
                    }
                },
                "required": ["title", "path"]
            }
        }
    }
]

# 本地 Mock 函数实现
def get_weather_mock(location):
    print(f"\n[🛠️ Tool Execution] Executing get_weather for location: {location}")
    return "Cloudy 20°C"

def create_article_mock(title, path, content=""):
    print(f"\n[🛠️ Tool Execution] Executing create_article:\n  Title: {title}\n  Path: {path}\n  Content Length: {len(content)}")
    return f"✅ Article created successfully at {path}"

# 将工具名映射到实际的 Python 函数
TOOL_CALL_MAP = {
    "get_weather": get_weather_mock,
    "create_article": create_article_mock
}

print("✅ 工具定义与本地实现准备完毕")

### 执行测试

In [ ]:
# 初始用户消息
messages = [
    {"role": "user", "content": "帮我创建一篇关于 deepseek 的文章，标题叫 'DeepSeekV3 的 MLA 和 MoE 详解'，放在 knowledge 目录下。你先告诉我你准备写什么，再调用工具创建文章。"}
]

print("\n" + "="*80)https://api-docs.deepseek.com/zh-cn/guides/tool_calls
print("🚀 开始请求 DeepSeek API (First Request)...")
print("="*80)

# 【一】第一次 API 请求：带上 tools 告诉 AI 它可以调用哪些工具
try:
    # 打印即将发送的请求体信息(为了清晰，这里手动提取一下相当于发过去的数据)
    request_summary = {
        "model": "deepseek-chat",
        "messages": messages,
        "tools_count": len(tools)
    }
    print(f"\n[📤 Request to DeepSeek (First)]\n{json.dumps(request_summary, indent=2, ensure_ascii=False)}")
    
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools,
        temperature=0.7 # 根据实际需要调整
    )
    
    # 打印完整的原始响应对象 (转换为 dict 打印)
    print(f"\n[📥 Raw Response from DeepSeek (First)]\n{response.model_dump_json(indent=2)}")
    
    # 提取 message 和 tool_calls
    message = response.choices[0].message
    tool_calls = message.tool_calls
    
    print(f"\n[💡 AI Message Content (First)]: {message.content}")
    
    if tool_calls is not None and len(tool_calls) > 0:
        print(f"\n[🛠️ Detected {len(tool_calls)} Tool Call(s)]")
        
        # ★★★ 关键步骤：把 AI 返回的带有 tool_calls 信息的 assistant message 也要加入历史记录 ★★★
        messages.append(message)
        
        # 【二】执行本地工具
        for tool in tool_calls:
            function_name = tool.function.name
            # 解析并打印工具参数
            try:
                arguments = json.loads(tool.function.arguments)
                print(f"  ➜ Function: {function_name}")
                print(f"  ➜ Arguments JSON parsed: {arguments}")
            except json.JSONDecodeError as e:
                print(f"  ❌ Failed to parse JSON arguments: {tool.function.arguments}")
                arguments = {}
            
            # 找到对应的本地实现并执行
            if function_name in TOOL_CALL_MAP:
                tool_function = TOOL_CALL_MAP[function_name]
                tool_result = tool_function(**arguments)
                print(f"  ➜ Execution Result: {tool_result}")
                
                # ★★★ 关键步骤：追加工具的执行结果到上下文中 ★★★
                # role 必须是 "tool"，并且需要对应的 tool_call_id
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool.id,
                    "content": tool_result
                })
            else:
                 print(f"  ❌ Tool {function_name} not found in TOOL_CALL_MAP")
                 messages.append({
                    "role": "tool",
                    "tool_call_id": tool.id,
                    "content": "Error: tool not found"
                })
                 
        print("\n" + "="*80)
        print("🚀 请求 DeepSeek API获取最终回复 (Second Request)...")
        print("="*80)
        
        # 打印第二次包含 tool 结果后的消息体
        print(f"\n[📤 Request Messages (Second)]\n{json.dumps(messages, default=lambda o: o.__dict__, indent=2, ensure_ascii=False)}")

        # 【三】第二次 API 请求：将含工具结果的面条发给 AI，获取最终评价
        final_response = client.chat.completions.create(
             model="deepseek-chat",
             messages=messages,
             temperature=0.7
        )
        
        print(f"\n[📥 Raw Response from DeepSeek (Second)]\n{final_response.model_dump_json(indent=2)}")
        print(f"\n[🎉 Final AI Output]:\n{final_response.choices[0].message.content}")

    else:
        print("\n⚠️ AI did NOT call any tools.")

except Exception as e:
    import traceback
    print("\n❌ [CRITICAL ERROR] Occurred during API call sequence!")
    print(f"Exception Type: {type(e).__name__}")
    print(f"Exception Message: {str(e)}")
    print("Traceback:")
    traceback.print_exc()